<a href="https://colab.research.google.com/github/Maddox159-crypto/ESAA_assignment/blob/main/%EB%88%84%EB%A9%94%EB%9D%BC%EC%9D%B4%EF%BC%92.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.path.exists('/content/drive/MyDrive/ESAA/numerai_r1300__v5_2_train.parquet')

True

In [ ]:
import pyarrow.parquet as pq
import numpy as np
from sklearn.decomposition import IncrementalPCA
import time

train_path = '/content/drive/MyDrive/ESAA/numerai_r1300__v5_2_train.parquet'
val_path   = '/content/drive/MyDrive/ESAA/numerai_r1300__v5_2_validation.parquet'

N_COMPONENTS = 300
BATCH_SIZE = 50000

feature_cols = [c for c in pq.ParquetFile(train_path).schema_arrow.names if c.startswith('feature')]
print(f"feature 개수: {len(feature_cols)}")

ipca_300 = IncrementalPCA(n_components=N_COMPONENTS, batch_size=BATCH_SIZE)

# ---- 1) Fit (train) ----
t0 = time.time()
pf = pq.ParquetFile(train_path)
n_batches = 0
for batch in pf.iter_batches(batch_size=BATCH_SIZE, columns=feature_cols):
    arr = batch.to_pandas().to_numpy(dtype=np.float32) / 4.0
    ipca_300.partial_fit(arr)
    n_batches += 1
    if n_batches % 10 == 0:
        elapsed = time.time() - t0
        print(f"batch {n_batches} 완료, 경과 {elapsed/60:.1f}분")

print(f"fit 완료, 총 {(time.time()-t0)/60:.1f}분")
print(f"설명분산: {ipca_300.explained_variance_ratio_.sum()*100:.2f}%")

# ---- 2) Transform (train) ----
t1 = time.time()
X_train_pca_300 = []
pf = pq.ParquetFile(train_path)
for batch in pf.iter_batches(batch_size=BATCH_SIZE, columns=feature_cols):
    arr = batch.to_pandas().to_numpy(dtype=np.float32) / 4.0
    X_train_pca_300.append(ipca_300.transform(arr))
X_train_pca_300 = np.vstack(X_train_pca_300)
print(f"train transform 완료, {(time.time()-t1)/60:.1f}분, shape={X_train_pca_300.shape}")

np.save('/content/drive/MyDrive/ESAA/X_train_pca_300.npy', X_train_pca_300)
del X_train_pca_300

# ---- 3) Transform (val) ----
t2 = time.time()
X_val_pca_300 = []
pf = pq.ParquetFile(val_path)
for batch in pf.iter_batches(batch_size=BATCH_SIZE, columns=feature_cols):
    arr = batch.to_pandas().to_numpy(dtype=np.float32) / 4.0
    X_val_pca_300.append(ipca_300.transform(arr))
X_val_pca_300 = np.vstack(X_val_pca_300)
print(f"val transform 완료, {(time.time()-t2)/60:.1f}분, shape={X_val_pca_300.shape}")

np.save('/content/drive/MyDrive/ESAA/X_val_pca_300.npy', X_val_pca_300)

print(f"전체 소요 시간: {(time.time()-t0)/60:.1f}분")

feature 개수: 2748
batch 10 완료, 경과 7.6분
batch 20 완료, 경과 15.2분
batch 30 완료, 경과 23.2분
batch 40 완료, 경과 30.8분
batch 50 완료, 경과 38.7분
fit 완료, 총 42.4분
설명분산: 73.34%
train transform 완료, 1.9분, shape=(2746268, 300)
val transform 완료, 5.1분, shape=(4071435, 300)
전체 소요 시간: 50.2분
